# RedConecta Telecom S.A. — Análisis Exploratorio de Datos (EDA)
**Autor:** Equipo de Ciencia de Datos  
**Objetivo:** Exploración inicial, auditoría de calidad de datos y evaluación preliminar de variables operacionales y SLA.

## 1. Importación de Librerías y Configuración del Entorno

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración de visualización y estilo
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## 2. Carga del Dataset y Estructura Básica

In [ ]:
# Ruta estándar del dataset en data/raw o raíz
ruta_posible_1 = '../data/raw/dataset_set_D_telecomunicaciones.xlsx'
ruta_posible_2 = 'dataset_set_D_telecomunicaciones.xlsx'

if os.path.exists(ruta_posible_1):
    ruta_dataset = ruta_posible_1
else:
    ruta_dataset = ruta_posible_2

df = pd.read_excel(ruta_dataset)
df['fecha_registro'] = pd.to_datetime(df['fecha_registro'])

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

## 3. Auditoría de Calidad de Datos
Revisamos completitud, tipos de datos, duplicados y consistencia general.

In [ ]:
# Información de tipos y memoria
print("--- Información del DataFrame ---")
df.info()

# Conteo de valores nulos
print("\n--- Conteo de Valores Nulos ---")
print(df.isnull().sum())

# Verificación de duplicados
duplicados = df.duplicated().sum()
print(f"\nFilas duplicadas exactas: {duplicados}")
print(f"Rango de fechas: {df['fecha_registro'].min().date()} al {df['fecha_registro'].max().date()}")

## 4. Estadísticas Descriptivas

In [ ]:
# Resumen de variables numéricas
df.describe()

In [ ]:
# Resumen de variables categóricas
print("Distribución por Región:")
print(df['region'].value_counts())
print("\nDistribución por Tipo de Servicio:")
print(df['tipo_servicio'].value_counts())
print("\nDistribución de Cumplimiento SLA:")
print(df['cumplimiento_sla'].value_counts(normalize=True) * 100)

## 5. Análisis Multidimensional: Región y Tipo de Servicio

In [ ]:
# Agrupación cruzada por Región y Tipo de Servicio
resumen_cruzado = df.groupby(['region', 'tipo_servicio']).agg(
    registros=('fecha_registro', 'count'),
    latencia_media_ms=('latencia_ms', 'mean'),
    resolucion_media_hr=('tiempo_resolucion_hr', 'mean'),
    clientes_afectados_prom=('clientes_afectados', 'mean'),
    cumplimiento_sla_pct=('cumplimiento_sla', lambda x: (x == 'Cumple').mean() * 100)
).reset_index()

resumen_cruzado

## 6. Visualización Exploratoria de Datos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de Latencia por Tipo de Servicio
sns.boxplot(data=df, x='tipo_servicio', y='latencia_ms', palette='Set2', ax=axes[0])
axes[0].set_title('Distribución de Latencia (ms) por Servicio', fontweight='bold')
axes[0].set_xlabel('Tipo de Servicio')
axes[0].set_ylabel('Latencia (ms)')

# Tasa de Cumplimiento de SLA por Región
region_sla = df.groupby('region')['cumplimiento_sla'].apply(lambda x: (x == 'Cumple').mean() * 100).reset_index()
sns.barplot(data=region_sla, x='region', y='cumplimiento_sla', palette='Blues_d', ax=axes[1])
axes[1].set_title('Cumplimiento de SLA (%) por Región', fontweight='bold')
axes[1].set_xlabel('Región')
axes[1].set_ylabel('% Cumple')
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 7. Análisis de Correlaciones Lineales

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
matriz_corr = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de Correlación Operacional', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Detección de Inconsistencias y Pruebas Preliminares
Evaluación de casos especiales (días sin incidencias pero con clientes afectados o tickets).

In [ ]:
# Inconsistencia: 0 incidencias técnicas pero con tickets o impacto a usuarios
cero_incidencias = df[df['incidencias_reportadas'] == 0]
print(f"Casos detectados con 0 incidencias técnicas: {len(cero_incidencias)}")
cero_incidencias[['fecha_registro', 'region', 'tipo_servicio', 'incidencias_reportadas', 'tickets_soporte', 'clientes_afectados', 'cumplimiento_sla']]

In [ ]:
# Prueba ANOVA preliminar con SciPy para tiempo de resolución por región
grupo_c = df[df['region'] == 'Centro']['tiempo_resolucion_hr']
grupo_n = df[df['region'] == 'Norte']['tiempo_resolucion_hr']
grupo_s = df[df['region'] == 'Sur']['tiempo_resolucion_hr']

f_stat, p_val = stats.f_oneway(grupo_c, grupo_n, grupo_s)
print(f"Prueba ANOVA (Tiempo de Resolución entre Regiones): F = {f_stat:.4f}, p = {p_val:.4f}")
if p_val < 0.05:
    print("Conclusión: Existen diferencias significativas entre regiones a un alfa del 5%.")
else:
    print("Conclusión: No se detectan diferencias estadísticamente significativas al 5% (tendencia marginal).")

## 9. Conclusiones y Próximos Pasos
- **Calidad del dato:** El dataset está completo (200 registros sin nulos), pero presenta incoherencias lógicas de negocio (días con 0 incidencias y cientos de clientes afectados) y falta de correlación lineal en variables clave.
- **Desempeño regional:** La Región Sur muestra el mayor cumplimiento de SLA (55.2%), pero los tiempos de resolución más elevados (~28.2 hrs).
- **Servicio crítico:** Internet Hogar concentra el mayor volumen de clientes impactados (>20,000 acumulados) y la tasa de cumplimiento más baja (46.8%).
- **Paso siguiente:** Modularizar funciones en `src/stats_engine.py` y `src/dashboard_exploratorio.py` para automatizar reportes a Soporte Técnico y Experiencia de Clientes.